# 9.4 Stacking (Stacked Generalisation)

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 9: Ensemble Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-09-ensemble-learning/09.04_stacking.ipynb)

Companion code for Section 9.4 of the book. Run the cells in order from top to bottom.

**Example 9.4.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import accuracy_score
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target
X_sc = StandardScaler().fit_transform(X)

# ── Stacking from scratch ─────────────────────────────────────────
def stacking_oof(X, y, base_models, meta_learner, n_splits=5):
    '''
    Stacked generalisation with out-of-fold meta-features.
    Returns OOF predictions, fitted base models (on full data), fitted meta-learner.
    '''
    n = len(X)
    k = len(base_models)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Meta-features matrix: rows=samples, cols=base models
    Z = np.zeros((n, k))

    print(f'Building OOF meta-features ({n_splits}-fold)...')
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr        = y[train_idx]

        for j, model in enumerate(base_models):
            m = clone(model)
            m.fit(X_tr, y_tr)
            # Use predict_proba if available, else predict
            if hasattr(m, 'predict_proba'):
                Z[val_idx, j] = m.predict_proba(X_val)[:, 1]
            else:
                Z[val_idx, j] = m.predict(X_val)

        print(f'  Fold {fold+1}/{n_splits} complete')

    # Retrain all base models on full training data
    fitted_base = []
    print('Retraining base models on full data...')
    for model in base_models:
        m = clone(model)
        m.fit(X, y)
        fitted_base.append(m)

    # Train meta-learner on OOF predictions
    print('Training meta-learner...')
    meta = clone(meta_learner)
    meta.fit(Z, y)

    # OOF accuracy
    oof_preds = (Z @ np.ones(k) / k >= 0.5).astype(int)  # Simple average
    meta_oof  = (meta.predict_proba(Z)[:,1] >= 0.5).astype(int)

    return fitted_base, meta, Z, meta_oof


# ── Define base models (DIVERSE!) ─────────────────────────────────
base_models = [
    LogisticRegression(max_iter=1000, random_state=42),
    RandomForestClassifier(n_estimators=100, random_state=42),
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    SVC(probability=True, kernel='rbf', random_state=42),
    KNeighborsClassifier(n_neighbors=7),
    GaussianNB(),
    ExtraTreesClassifier(n_estimators=100, random_state=42),
]

meta_learner = LogisticRegression(max_iter=1000, C=1.0, random_state=42)

# ── Run stacking ──────────────────────────────────────────────────
fitted_base, meta, Z_oof, meta_preds = stacking_oof(
    X_sc, y, base_models, meta_learner, n_splits=5
)

print('\n=== Stacking Results ===')
print(f'\n{"Model":<25} {"CV Accuracy":>13} {"OOF Accuracy":>14}')
print('='*55)

# Individual base model CV scores
model_names = ['LogReg','RandomForest','GradBoost','SVM','KNN','NaiveBayes','ExtraTrees']
for name, model in zip(model_names, base_models):
    cv = cross_val_score(clone(model), X_sc, y, cv=5, scoring='accuracy').mean()
    print(f'{name:<25} {cv:>13.4f}')

# Meta-learner OOF accuracy
meta_oof_acc = accuracy_score(y, meta_preds)
print(f'{'STACK (Meta-Learner)':<25} {"":>13} {meta_oof_acc:>14.4f}')

# sklearn StackingClassifier (easier API)
estimators = [(n, clone(m)) for n, m in zip(model_names, base_models)]
sk_stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5, passthrough=False, n_jobs=-1
)
sk_cv = cross_val_score(sk_stack, X_sc, y, cv=5, scoring='accuracy')
print(f'{'STACK (sklearn)':<25} {sk_cv.mean():>13.4f} ± {sk_cv.std():>.4f}')

# ── Meta-feature correlation ──────────────────────────────────────
print('\nOOF Meta-Feature Correlation (should be moderate, not 1.0):')
Z_df = pd.DataFrame(Z_oof, columns=model_names)
print(Z_df.corr().round(3))

# ── Visualise meta-feature space ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Stacking Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
# Show LogReg vs RF meta-features coloured by truth
colors = ['#E53935' if c==0 else '#1976D2' for c in y]
ax.scatter(Z_oof[:,0], Z_oof[:,1], c=colors, s=15, alpha=0.7, edgecolors='none')
ax.set_xlabel('LogReg OOF Prediction (P(y=1))')
ax.set_ylabel('RandomForest OOF Prediction (P(y=1))')
ax.set_title('Meta-Feature Space\n(LogReg vs RandomForest OOF probabilities)')
ax.grid(alpha=0.3)

ax = axes[1]
# Model comparison
names_plot = model_names + ['Stack']
scores_plot= [cross_val_score(clone(m),X_sc,y,cv=5).mean() for m in base_models] + [sk_cv.mean()]
colors_bar = ['#BDBDBD']*len(base_models) + ['#1976D2']
bars = ax.bar(names_plot, scores_plot, color=colors_bar, edgecolor='white')
for bar, sc in zip(bars, scores_plot):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
           f'{sc:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('5-Fold CV Accuracy'); ax.set_title('Individual vs Stacked Performance')
ax.set_xticklabels(names_plot, rotation=45, ha='right', fontsize=9)
ax.set_ylim(0.90, 1.00); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('stacking_analysis.png', dpi=150, bbox_inches='tight')
plt.show()